1. Đọc một mẫu có đáp án và một mẫu không có đáp án

In [1]:
import json
from pathlib import Path

# Nếu notebook đang ở thư mục notebooks/, lùi lên thư mục gốc dự án.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

train_path = project_root / "data" / "processed" / "train.json"

answerable_example = None
unanswerable_example = None

# Tìm mỗi loại một mẫu trong tập train con.
with train_path.open("r", encoding="utf-8") as input_file:
    for line in input_file:
        example = json.loads(line)
        has_answer = len(example["answers"]["text"]) > 0

        if not example["is_impossible"] and has_answer:
            answerable_example = example

        if example["is_impossible"] or not has_answer:
            unanswerable_example = example

        if answerable_example is not None and unanswerable_example is not None:
            break

print("Mẫu có đáp án:", answerable_example["answers"]["text"][0])
print("Mẫu không có đáp án:", unanswerable_example["is_impossible"])

Mẫu có đáp án: hoạt động văn chương, âm nhạc, nghệ thuật và sân khấu
Mẫu không có đáp án: True


2. Tải fast tokenizer và import các hàm

In [2]:
import sys
from pathlib import Path
from transformers import AutoTokenizer

# project_root đã được xác định ở bước 1; dùng được khi chuyển dự án sang máy khác.
# Kiểm tra Python có nhìn thấy file cần import không.
preprocessing_file = project_root / "src" / "data_preprocessing.py"
print("Đang tìm file:", preprocessing_file)
assert preprocessing_file.exists(), "Không tìm thấy data_preprocessing.py ở đường dẫn này."

# Thêm thư mục gốc dự án vào danh sách nơi Python tìm module.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Bây giờ Python có thể tìm package src.
from src.data_preprocessing import (
    prepare_train_features,
    prepare_validation_features,
)

#khởi tạo tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "xlm-roberta-base",
    use_fast=True,
)


print("Import thành công")
print("Tokenizer đã tải:", tokenizer.name_or_path)
print("Fast tokenizer:", tokenizer.is_fast)

c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang tìm file: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\src\data_preprocessing.py
Import thành công
Tokenizer đã tải: xlm-roberta-base
Fast tokenizer: True


3. Kiểm tra ánh xạ đáp án từ ký tự sang token

In [3]:
# Hàm nhận một mẫu riêng lẻ nhưng đóng gói các trường thành batch một phần tử.
def make_batch(example):
    return {
        "id": [example["id"]],
        "question": [example["question"].lstrip()],
        "context": [example["context"]],
        "answers": [example["answers"]],
        "is_impossible": [example["is_impossible"]],
    }

answerable_batch = make_batch(answerable_example)

# Gọi hàm tiền xử lý để lấy nhãn token bắt đầu và kết thúc.
train_features = prepare_train_features(answerable_batch, tokenizer)

# Tokenize lại cùng mẫu để xem offset ký tự đối chiếu với nhãn.
raw_features = tokenizer(
    answerable_batch["question"],
    answerable_batch["context"],
    truncation="only_second",
    max_length=384,
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding="max_length",
)

answer = answerable_example["answers"]["text"][0]
answer_start = answerable_example["answers"]["answer_start"][0]
answer_end = answer_start + len(answer)

# Xác nhận offset ký tự trong dữ liệu trích đúng chuỗi đáp án.
assert answerable_example["context"][answer_start:answer_end] == answer

# Tìm cửa sổ có nhãn đáp án thay vì nhãn CLS.
answer_span_found = False

for feature_index, (start_token, end_token) in enumerate(
    zip(train_features["start_positions"], train_features["end_positions"])
):
    input_ids = raw_features["input_ids"][feature_index]
    cls_index = input_ids.index(tokenizer.cls_token_id)

    # Nhãn CLS nghĩa là cửa sổ này không chứa trọn đáp án.
    if start_token == cls_index and end_token == cls_index:
        continue

    offsets = raw_features["offset_mapping"][feature_index]
    sequence_ids = raw_features.sequence_ids(feature_index)

    # Hai đầu của đáp án phải thuộc context.
    assert sequence_ids[start_token] == 1
    assert sequence_ids[end_token] == 1

    # Offset token phải bao phủ khoảng ký tự của đáp án.
    assert offsets[start_token][0] <= answer_start
    assert offsets[end_token][1] >= answer_end

    answer_span_found = True
    print("Đáp án chuẩn:", answer)
    print("Chỉ số token bắt đầu:", start_token)
    print("Chỉ số token kết thúc:", end_token)
    print("Offset token bắt đầu:", offsets[start_token])
    print("Offset token kết thúc:", offsets[end_token])
    break

# Một mẫu có đáp án phải được tìm thấy trong ít nhất một cửa sổ.
assert answer_span_found
print("Kiểm tra ánh xạ đáp án: đạt")

Đáp án chuẩn: hoạt động văn chương, âm nhạc, nghệ thuật và sân khấu
Chỉ số token bắt đầu: 55
Chỉ số token kết thúc: 67
Offset token bắt đầu: (179, 183)
Offset token kết thúc: (228, 232)
Kiểm tra ánh xạ đáp án: đạt


4. Kiểm tra câu không có đáp án và validation

In [4]:
# Với câu không có đáp án, mọi cửa sổ phải mang nhãn CLS ở cả hai đầu.
unanswerable_batch = make_batch(unanswerable_example)
no_answer_features = prepare_train_features(unanswerable_batch, tokenizer)

for feature_index, input_ids in enumerate(no_answer_features["input_ids"]):
    cls_index = input_ids.index(tokenizer.cls_token_id)

    assert no_answer_features["start_positions"][feature_index] == cls_index
    assert no_answer_features["end_positions"][feature_index] == cls_index

print("Kiểm tra nhãn câu không có đáp án: đạt")

# Validation phải giữ ID mẫu và offset của context.
validation_features = prepare_validation_features(answerable_batch, tokenizer)

assert all(
    example_id == answerable_example["id"]
    for example_id in validation_features["example_id"]
)

for offsets in validation_features["offset_mapping"]:
    assert any(offset is not None for offset in offsets)

print("Kiểm tra metadata validation: đạt")

Kiểm tra nhãn câu không có đáp án: đạt
Kiểm tra metadata validation: đạt


## Bước 5: Kiểm tra các trường hợp biên với XLM-R và mDeBERTa

Kiểm tra hai fast tokenizer trên câu có đáp án, không có đáp án, nhiều đáp án chuẩn và đáp án vắt qua ranh giới cửa sổ. Khi huấn luyện, nhãn dùng đáp án chuẩn đầu tiên. Các đáp án còn lại vẫn được giữ để chấm EM/F1. Cell này chỉ dùng tokenizer.

In [5]:
from copy import deepcopy

# Cố tình dùng cửa sổ nhỏ để thấy được một đáp án vắt qua ranh giới.
CHECK_MAX_LENGTH = 48
CHECK_STRIDE = 12


def make_check_example(example_id, context, answers, is_impossible=False):
    """Tạo mẫu QA và ghi offset từ chính context dùng để kiểm tra."""
    return {
        "id": example_id,
        "question": "   Địa danh nào?",
        "context": context,
        "answers": {
            "text": answers,
            "answer_start": [context.index(answer) for answer in answers],
        },
        "is_impossible": is_impossible,
    }


def check_window_labels(example, fast_tokenizer, *, require_boundary=False):
    # Giữ nguyên khoảng trắng đầu câu hỏi để phát hiện thay đổi batch ngoài ý muốn.
    batch = {key: [value] for key, value in example.items()}
    before = deepcopy(batch)
    train = prepare_train_features(batch, fast_tokenizer, CHECK_MAX_LENGTH, CHECK_STRIDE)
    assert batch == before, "Train đã thay đổi batch đầu vào."
    validation = prepare_validation_features(
        batch, fast_tokenizer, CHECK_MAX_LENGTH, CHECK_STRIDE
    )
    assert batch == before, "Validation đã thay đổi batch đầu vào."

    # Hai hàm phải tạo cùng cửa sổ và giữ một nhãn train cho mỗi feature.
    assert train["input_ids"] == validation["input_ids"]
    feature_count = len(train["input_ids"])
    assert feature_count == len(train["start_positions"]) == len(train["end_positions"])
    assert validation["example_id"] == [example["id"]] * feature_count

    found_answer = False
    for feature_index, input_ids in enumerate(train["input_ids"]):
        cls_index = input_ids.index(fast_tokenizer.cls_token_id)
        start_token = train["start_positions"][feature_index]
        end_token = train["end_positions"][feature_index]
        is_cls = (start_token, end_token) == (cls_index, cls_index)

        # Cửa sổ đầu cắt ngang đáp án. Cửa sổ sau có vùng chồng lặp để tìm lại.
        if require_boundary and feature_index == 0:
            assert is_cls, "Cửa sổ đầu chỉ chứa một phần đáp án."
        if example.get("is_impossible", False):
            assert is_cls, "Câu không có đáp án phải trỏ về CLS."
        if is_cls:
            continue

        # Offset của hai token phải thuộc context và bao phủ đáp án đầu tiên.
        offsets = validation["offset_mapping"][feature_index]
        assert start_token <= end_token
        assert offsets[start_token] is not None and offsets[end_token] is not None
        answer_start = example["answers"]["answer_start"][0]
        answer_end = answer_start + len(example["answers"]["text"][0])
        assert offsets[start_token][0] <= answer_start
        assert offsets[end_token][1] >= answer_end
        found_answer = True

    assert found_answer == (not example.get("is_impossible", False))
    print(f"Đạt: {example['id']} ({feature_count} cửa sổ)")


for checkpoint in ("xlm-roberta-base", "microsoft/mdeberta-v3-base"):
    fast_tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
    assert fast_tokenizer.is_fast
    print(f"\nTokenizer: {checkpoint}")

    check_window_labels(
        make_check_example("có đáp án", "Hà Nội nằm ở miền Bắc.", ["Hà Nội"]),
        fast_tokenizer,
    )
    check_window_labels(
        make_check_example("không có đáp án", "Hà Nội có nhiều hồ.", [], True),
        fast_tokenizer,
    )
    check_window_labels(
        make_check_example("nhiều đáp án chuẩn", "Hà Nội và Huế đều có di tích.", ["Hà Nội", "Huế"]),
        fast_tokenizer,
    )

    # Đo ranh giới bằng offset thật của tokenizer hiện tại.
    units = ["mây"] * 100
    long_context = " ".join(units)
    edge_example = make_check_example("đáp án qua ranh giới", long_context, [])
    edge_example.pop("is_impossible")  # Cũng kiểm tra đường mặc định khi thiếu cột.
    probe = {key: [value] for key, value in edge_example.items()}
    windows = prepare_validation_features(probe, fast_tokenizer, CHECK_MAX_LENGTH, CHECK_STRIDE)
    assert len(windows["offset_mapping"]) > 1
    first_context_offsets = [offset for offset in windows["offset_mapping"][0] if offset is not None]
    first_end = max(offset[1] for offset in first_context_offsets)

    # Context gồm các cụm "mây " cách đều nhau. Lấy cụm tại ranh giới và cụm kế tiếp.
    step = len("mây ")
    answer_start = ((first_end - 1) // step) * step
    answer_end = answer_start + len("mây mây")
    assert answer_start < first_end < answer_end
    edge_example["answers"] = {
        "text": [long_context[answer_start:answer_end]],
        "answer_start": [answer_start],
    }
    check_window_labels(edge_example, fast_tokenizer, require_boundary=True)

    # Có offset nhưng thiếu text phải báo rõ dữ liệu sai thay vì truy cập [0].
    bad = make_check_example("thiếu text", "Hà Nội có nhiều hồ.", ["Hà Nội"])
    bad["answers"]["text"] = []
    try:
        prepare_train_features(
            {key: [value] for key, value in bad.items()}, fast_tokenizer,
            CHECK_MAX_LENGTH, CHECK_STRIDE,
        )
    except ValueError as error:
        assert "cùng số phần tử" in str(error)
        print("Đạt: phát hiện text và answer_start lệch nhau")
    else:
        raise AssertionError("Dữ liệu thiếu text phải báo lỗi.")

print("\nT07: cả hai tokenizer đều đạt.")


Tokenizer: xlm-roberta-base
Đạt: có đáp án (1 cửa sổ)
Đạt: không có đáp án (1 cửa sổ)
Đạt: nhiều đáp án chuẩn (1 cửa sổ)
Đạt: đáp án qua ranh giới (2 cửa sổ)
Đạt: phát hiện text và answer_start lệch nhau


c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\havuh\.cache\huggingface\hub\models--microsoft--mdeberta-v3-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\havuh\OneDrive\Deskto


Tokenizer: microsoft/mdeberta-v3-base
Đạt: có đáp án (1 cửa sổ)
Đạt: không có đáp án (1 cửa sổ)
Đạt: nhiều đáp án chuẩn (1 cửa sổ)
Đạt: đáp án qua ranh giới (2 cửa sổ)
Đạt: phát hiện text và answer_start lệch nhau

T07: cả hai tokenizer đều đạt.


-----------------------